In [1]:
import numpy as np
import pandas as pd
from scipy.signal import welch
from sklearn.linear_model import LinearRegression

# ----------------------------------
# PARAMETERS
# ----------------------------------
fs = 200
window_sec = 1
win_size = fs * window_sec   # 200 samples

In [ ]:
df=pd.read_csv('data/emg_data_600.csv') 

In [5]:
def compute_rms(signal):
    return np.sqrt(np.mean(signal ** 2))

def compute_mdf(signal, fs):
    f, Pxx = welch(signal, fs=fs, nperseg=len(signal))
    cumsum = np.cumsum(Pxx)
    return f[np.where(cumsum >= cumsum[-1] / 2)[0][0]]

# ----------------------------------
# WINDOWED FEATURE EXTRACTION
# ----------------------------------
step = win_size
n_windows = (len(df) - win_size) // step + 1

rms_list = []
mdf_list = []

for w in range(n_windows):
    start = w * step
    raw_win = df["Raw_EMG"].iloc[start:start+win_size].values
    env_win = df["Envelope_EMG"].iloc[start:start+win_size].values

    rms_list.append(compute_rms(env_win))
    mdf_list.append(compute_mdf(raw_win, fs))

rms_df = pd.Series(rms_list, name="RMS")
mdf_df = pd.Series(mdf_list, name="MDF")

print("Total windows:", len(rms_df))

# ----------------------------------
# TREND (SLOPE) COMPUTATION
# ----------------------------------
time_windows = np.arange(len(rms_df)).reshape(-1, 1)

rms_slope = LinearRegression().fit(time_windows, rms_df.values).coef_[0]
mdf_slope = LinearRegression().fit(time_windows, mdf_df.values).coef_[0]

print(f"RMS slope: {rms_slope:.6f}")
print(f"MDF slope: {mdf_slope:.6f}")

# ----------------------------------
# FATIGUE DETECTION FUNCTION
# ----------------------------------
def detect_fatigue(new_raw, new_env, rms_slope, mdf_slope, rms_df, mdf_df, fs):
    rms_val = compute_rms(new_env)
    mdf_val = compute_mdf(new_raw, fs)

    mean_rms = rms_df.mean()
    mean_mdf = mdf_df.mean()

    if (
        rms_slope > 0 and
        mdf_slope < 0 and
        rms_val > mean_rms and
        mdf_val < mean_mdf
    ):
        return "Fatigued"
    else:
        return "Non-Fatigued"

# ----------------------------------
# EXAMPLE: NEW 1-SECOND INPUT
# ----------------------------------
start_sample = 116058   # choose any valid index
end_sample = start_sample + win_size

new_raw = df["Raw_EMG"].iloc[start_sample:end_sample].values
new_env = df["Envelope_EMG"].iloc[start_sample:end_sample].values

status = detect_fatigue(
    new_raw,
    new_env,
    rms_slope,
    mdf_slope,
    rms_df,
    mdf_df,
    fs
)

# ----------------------------------
# RESULT
# ----------------------------------
print("\n=== Fatigue Status ===")
print(status)

Total windows: 602
RMS slope: 0.098220
MDF slope: -0.000267

=== Fatigue Status ===
Fatigued
